In [ ]:
!pip install opendatasets --quiet
import opendatasets as od

# Replace this with your Kaggle dataset URL
dataset_url = 'https://www.kaggle.com/datasets/thoughtvector/customer-support-on-twitter'

# This will prompt for your kaggle.json credentials
od.download(dataset_url)

# Typically the data is downloaded into a folder named after the dataset
import pandas as pd
df = pd.read_csv('/content/customer-support-on-twitter/twcs/twcs.csv')

Skipping, found downloaded files in "./customer-support-on-twitter" (use force=True to force download)


In [ ]:
!pip install vaderSentiment
import numpy as np
import kagglehub
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.metrics import classification_report, f1_score
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Shuffle and take 20% of the data
df = df.sample(frac=0.2, random_state=42).copy()

# Reset index (optional, but recommended)
df.reset_index(drop=True, inplace=True)


print(f"Sampled size: {len(df)}")

Sampled size: 562355


In [ ]:
# Keep only customer tweets (inbound = True)
df = df[df['inbound'] == True].copy()

# Drop rows with missing text
df = df.dropna(subset=['text'])

# Create a cleaned text column (lowercase, remove URLs/mentions, keep punctuation)
def clean_tweet(text):
    import re
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)      # remove URLs
    text = re.sub(r'@\w+', '', text)         # remove @mentions
    text = re.sub(r'#', '', text)            # remove hash symbols
    text = re.sub(r'[^a-zA-Z0-9\s!?.]', '', text)  # keep letters, numbers, space, ! ? .
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['cleaned_text'] = df['text'].apply(clean_tweet)

# Remove very short tweets (optional)
df = df[df['cleaned_text'].str.len() >= 3]
df.reset_index(drop=True, inplace=True)

In [ ]:
analyzer = SentimentIntensityAnalyzer()

def urgent_v2(original_text, cleaned_text):
    score = 0

    # Keywords
    if any(w in cleaned_text for w in ['refund','broken','cancel','down','help','issue','problem','error']):
        score += 1

    # Sentiment: if neutral/positive → not urgent
    comp = analyzer.polarity_scores(cleaned_text)['compound']
    if comp >= 0:
        return False
    if comp < -0.2:
        score += 1

    # Exclamation
    if cleaned_text.count('!') >= 1:
        score += 1

    # Caps ratio (from original text)
    caps_ratio = sum(1 for c in original_text if c.isupper()) / max(len(original_text), 1)
    if caps_ratio > 0.1:
        score += 1

    # Short text
    if len(cleaned_text) < 50:
        score += 1

    # Question mark
    if '?' in cleaned_text:
        score += 1

    return score >= 2



In [ ]:
df['is_urgent'] = df.apply(lambda row: urgent_v2(row['text'], row['cleaned_text']), axis=1)

In [ ]:
df.head()

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id,cleaned_text,is_urgent
0,192624,161253,True,Wed Oct 04 13:59:33 +0000 2017,@161252 What's that egg website people talk about,192623,192625.0,whats that egg website people talk about,False
1,738238,296574,True,Fri Oct 06 18:29:06 +0000 2017,Why!🤷🏻‍♀️ #iOS11 @AppleSupport https://t.co/BX...,738237,NaN,why! ios11,False
2,1793929,539096,True,Thu Oct 12 06:04:41 +0000 2017,@331912 @115955 Thats better than having an un...,1793928,1793930.0,thats better than having an unstable connectio...,False
3,2088018,617376,True,Mon Nov 06 20:30:49 +0000 2017,@VirginAmerica is probably one of the best air...,2088017,NaN,is probably one of the best airlines ive ever ...,False
4,1941217,577298,True,Sun Oct 29 23:13:33 +0000 2017,@AlaskaAir Phone system seems to be down,1941219,1941216.0,phone system seems to be down,False


In [ ]:
print(df['is_urgent'].value_counts(normalize=True))

is_urgent
False    0.812249
True     0.187751
Name: proportion, dtype: float64


In [ ]:
X = df[['cleaned_text']]   # keep as DataFrame to preserve index
y = df['is_urgent']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.40, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f"Train size: {len(X_train)}, Val size: {len(X_val)}, Test size: {len(X_test)}")

Train size: 181456, Val size: 60485, Test size: 60486


In [ ]:
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2),
    stop_words='english',
    min_df=5,
    max_df=0.8
)

X_train_tfidf = vectorizer.fit_transform(X_train['cleaned_text'])
X_val_tfidf = vectorizer.transform(X_val['cleaned_text'])
X_test_tfidf = vectorizer.transform(X_test['cleaned_text'])

In [ ]:
X_train['cleaned_text']

,cleaned_text
101926,perfect. thank you so much.
145739,need support but cant update my email addy bc ...
211997,my xbox live just stopped working and i paid f...
193858,he tenido un problema...compre el rainbow six ...
94468,what about the services to windsor? trains can...
...,...
255281,still no one lifted the wires as promised can ...
192133,thank you so much !
287644,put simply..... these are amazing.
15814,well your automated help desk says ive placed ...


In [ ]:
"""
models = {
    'Logistic Regression': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(class_weight='balanced', n_estimators=100, random_state=42),
    'XGBoost': xgb.XGBClassifier(
        scale_pos_weight=(y_train==0).sum()/(y_train==1).sum(),
        eval_metric='logloss',
        random_state=42
    )
}
"""
models = {
    'XGBoost': xgb.XGBClassifier(
        scale_pos_weight=(y_train==0).sum()/(y_train==1).sum(),
        eval_metric='logloss',
        random_state=42
    )
}

results = {}
for name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    y_val_pred = model.predict(X_val_tfidf)
    f1 = f1_score(y_val, y_val_pred)
    results[name] = {'model': model, 'f1': f1}
    print(f"\n{name} - Validation F1: {f1:.4f}")
    print(classification_report(y_val, y_val_pred))


XGBoost - Validation F1: 0.5711
              precision    recall  f1-score   support

       False       0.91      0.85      0.88     49129
        True       0.50      0.66      0.57     11356

    accuracy                           0.81     60485
   macro avg       0.71      0.75      0.73     60485
weighted avg       0.84      0.81      0.82     60485



In [ ]:
best_name = max(results, key=lambda x: results[x]['f1'])
best_model = results[best_name]['model']
y_test_pred = best_model.predict(X_test_tfidf)
test_f1 = f1_score(y_test, y_test_pred)

print(f"\nBest TF-IDF model: {best_name}")
print(f"Test F1: {test_f1:.4f}")
print(classification_report(y_test, y_test_pred))


Best TF-IDF model: XGBoost
Test F1: 0.5746
              precision    recall  f1-score   support

       False       0.92      0.85      0.88     49130
        True       0.51      0.66      0.57     11356

    accuracy                           0.82     60486
   macro avg       0.71      0.76      0.73     60486
weighted avg       0.84      0.82      0.83     60486



In [ ]:
import joblib
joblib.dump(best_model, 'tfidf_priority_model.pkl')
joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')

['tfidf_vectorizer.pkl']